# Leakage-safe Titanic-style classification with Ginsu

This offline tutorial uses a deterministic **synthetic passenger dataset** with Titanic-style fields. It does not download or reproduce the historical Titanic dataset. The synthetic fixture keeps the notebook reproducible while demonstrating the production workflow.

We use three disjoint partitions:

1. **model training** fits the classifier;
2. **slice discovery** computes held-out per-row loss, fits error-independent discretization, and discovers candidate rules;
3. **fixed-rule validation** evaluates those unchanged rules on untouched rows.

Discovery and validation results are descriptive. They are not causal claims or automatic statistical-significance statements.

In [ ]:
import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from ginsu import CategoryPolicy, DiscretizationPlan, QuantileBins, Slicefinder
from ginsu.plotting import plot_error_dependence, plot_impact, plot_search_report


def take_rows(frame: pl.DataFrame, indices: np.ndarray) -> pl.DataFrame:
    return frame.gather(sorted(int(index) for index in indices))


def model_matrix(frame: pl.DataFrame) -> np.ndarray:
    return frame.select(
        pl.col("pclass").cast(pl.Float64),
        (pl.col("sex") == "female").cast(pl.Float64).alias("female"),
        pl.col("age"),
        pl.col("sibsp").cast(pl.Float64),
        pl.col("parch").cast(pl.Float64),
        pl.col("fare"),
        (pl.col("embarked") == "C").cast(pl.Float64).alias("embarked_c"),
        (pl.col("embarked") == "Q").cast(pl.Float64).alias("embarked_q"),
    ).to_numpy()

## Build a deterministic offline fixture

The outcome has a known relationship with passenger class, sex, age, family size, and embarkation. Randomness is seeded, and the generated table is Polars from the start.

In [ ]:
rng = np.random.default_rng(42)
row_count = 900
pclass = rng.choice([1, 2, 3], row_count, p=[0.22, 0.24, 0.54])
sex = rng.choice(["female", "male"], row_count, p=[0.36, 0.64])
age = np.clip(rng.normal(31 - 2 * (pclass - 2), 14, row_count), 1, 80)
sibsp = np.minimum(rng.poisson(0.5, row_count), 4)
parch = np.minimum(rng.poisson(0.4, row_count), 3)
fare = np.exp(rng.normal(3.1 - 0.55 * (pclass - 1), 0.55, row_count))
embarked = rng.choice(["C", "Q", "S"], row_count, p=[0.2, 0.1, 0.7])

log_odds = (
    1.5
    - 1.5 * (sex == "male")
    - 0.8 * (pclass == 3)
    - 0.025 * (age - 30)
    - 0.25 * (sibsp + parch >= 3)
    + 0.35 * (embarked == "C")
)
survival_probability = 1 / (1 + np.exp(-log_odds))
survived = rng.binomial(1, survival_probability)

passengers = pl.DataFrame(
    {
        "pclass": pl.Series(pclass).cast(pl.String),
        "sex": sex,
        "age": age,
        "sibsp": pl.Series(sibsp).cast(pl.String),
        "parch": pl.Series(parch).cast(pl.String),
        "fare": fare,
        "embarked": embarked,
    }
)
passengers.head()

## Fit the model and compute out-of-training losses

The model never sees the discovery or validation rows during fitting. The discovery and validation loss vectors are element-wise binary log loss, not aggregate scores.

In [ ]:
all_rows = np.arange(row_count)
model_rows, analysis_rows = train_test_split(
    all_rows, test_size=0.40, random_state=42, stratify=survived
)
discovery_rows, validation_rows = train_test_split(
    analysis_rows,
    test_size=0.50,
    random_state=43,
    stratify=survived[analysis_rows],
)
model_rows = np.sort(model_rows)
discovery_rows = np.sort(discovery_rows)
validation_rows = np.sort(validation_rows)

model = RandomForestClassifier(
    n_estimators=160, min_samples_leaf=8, random_state=42, n_jobs=1
)
model.fit(model_matrix(take_rows(passengers, model_rows)), survived[model_rows])

discovery = take_rows(passengers, discovery_rows)
validation = take_rows(passengers, validation_rows)
discovery_probability = np.clip(
    model.predict_proba(model_matrix(discovery))[:, 1], 1e-9, 1 - 1e-9
)
validation_probability = np.clip(
    model.predict_proba(model_matrix(validation))[:, 1], 1e-9, 1 - 1e-9
)
discovery_target = survived[discovery_rows]
validation_target = survived[validation_rows]
discovery_errors = -(
    discovery_target * np.log(discovery_probability)
    + (1 - discovery_target) * np.log(1 - discovery_probability)
)
validation_errors = -(
    validation_target * np.log(validation_probability)
    + (1 - validation_target) * np.log(1 - validation_probability)
)

pl.DataFrame(
    {
        "partition": ["model training", "slice discovery", "fixed-rule validation"],
        "rows": [len(model_rows), len(discovery_rows), len(validation_rows)],
    }
)

## Fit error-independent discretization and discover slices

Ginsu treats each input value as categorical. We therefore learn quantile boundaries for continuous columns from the discovery features only. The loss vector is not used to choose boundaries. The fitted plan is then reused unchanged for validation.

In [ ]:
binning = DiscretizationPlan(
    numeric={"age": QuantileBins(5), "fare": QuantileBins(5)},
    categorical={
        "pclass": CategoryPolicy(max_categories=3),
        "sex": CategoryPolicy(max_categories=2),
        "sibsp": CategoryPolicy(max_categories=5),
        "parch": CategoryPolicy(max_categories=4),
        "embarked": CategoryPolicy(max_categories=3),
    },
).fit(discovery)
discovery_binned = binning.transform(discovery)
validation_binned = binning.transform(validation)

finder = Slicefinder(
    alpha=0.95, k=8, max_l=2, min_sup=0.05, verbose=False
).fit(discovery_binned, discovery_errors)
finder.slice_statistics_.select(
    "rank", "support_fraction", "error_lift", "predicate_count"
).head(10)

## Validate unchanged rules and build an auditable compact view

Validation preserves discovery order and does not search again. Diversity selection uses validation membership only to create a reporting view; it never overwrites the fitted ranking.

In [ ]:
fixed_rule_validation = finder.validate_slices(
    validation_binned, validation_errors, min_support=0.05
)
diverse = finder.select_slices(
    validation_binned, method="diverse", k=5, max_jaccard=0.80
)
membership = finder.membership_frame(validation_binned)

fixed_rule_validation.statistics.select(
    "discovery_rank",
    "__ginsu_rule",
    "validation_status",
    "validation_support_fraction",
    "validation_error_lift",
    "error_lift_delta",
).head(10)

## Plot discovery impact, observed dependence, and search cost

The dependence plot is descriptive association, not causal or conventional partial dependence. The search profile is execution evidence, not model-quality evidence. In an interactive session, call ``figure.show()`` on any figure below.

In [ ]:
impact_figure = plot_impact(finder)
dependence_figure = plot_error_dependence(
    finder, discovery_binned, discovery_errors, feature="age"
)
search_figure = plot_search_report(finder.search_report_)

pl.DataFrame(
    {
        "figure": ["impact", "observed error dependence", "search profile"],
        "trace_count": [
            len(impact_figure.data),
            len(dependence_figure.data),
            len(search_figure.data),
        ],
    }
)

## Interpretation checklist

- Start with ``fixed_rule_validation.statistics``, not discovery score alone.
- Inspect ``validation_status`` and support before interpreting lift.
- Use ``diverse.decisions`` to audit overlap exclusions and ``membership`` for row-level review.
- Treat this synthetic example as an API tutorial. A real analysis still needs representative sampling, domain review, privacy controls, and a validation design appropriate to its deployment population.